# 08 — Acquire Renewables.ninja Profiles

## Research purpose

Acquire annual solar and wind profiles for the representative query locations and combine them into an hourly renewable-availability proxy.


## Imports

Load the libraries for API requests, response validation and table processing.


In [1]:
from hashlib import sha256
from io import StringIO
import json
import os
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
import requests

## Project paths

Locate the project folder and define the paths for query locations, the saved request plan, raw responses and the hourly renewable table.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "notebooks").is_dir() or not (PROJECT_ROOT / "data").is_dir():
    raise RuntimeError("Launch Jupyter from the folder containing data/ and notebooks/, or from its notebooks/ directory.")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"

QUERY_LOCATIONS_PATH = DATA_INTERIM / "renewables_ninja_query_locations.csv"
REQUEST_PLAN_PATH = DATA_INTERIM / "renewables_ninja_request_plan.csv"
HOURLY_OUTPUT_PATH = DATA_INTERIM / "renewable_availability_cluster_hourly.parquet"
RAW_CACHE_DIR = DATA_RAW / "renewables"

## Renewable and acquisition settings

Select the 2023–2025 MERRA-2 profiles and specify the PV and wind configurations, including PV losses and orientation, wind hub height and turbine model. Set request limits, retry controls and timeouts, with forced downloads and failed-request retries disabled by default.


In [3]:
# Weather and renewable-profile assumptions
WEATHER_YEARS = [2023, 2024, 2025]
TECHNOLOGIES = ["pv", "wind"]
PV_PARAMS = {
    "dataset": "merra2",
    "capacity": 1,
    "system_loss": 0.1,
    "tracking": 0,
    "tilt": 35,
    "azim": 180,
    "raw": "false",
    "local_time": "true",
    "header": "true",
    "format": "csv",
}

WIND_PARAMS = {
    "dataset": "merra2",
    "capacity": 1,
    "height": 100,
    "turbine": "Vestas V90 2000",
    "raw": "false",
    "local_time": "true",
    "header": "true",
    "format": "csv",
}

# Acquisition controls
FORCE_DOWNLOAD = False
RETRY_FAILED = False
HOURLY_REQUEST_LIMIT = 50
MIN_SECONDS_BETWEEN_REQUESTS = 1.1
AUTO_WAIT_ON_RATE_LIMIT = True
RATE_LIMIT_WAIT_BUFFER_SECONDS = 75
MAX_NEW_REQUESTS_PER_RUN = None  # Set to 50 for manual hourly batches. None lets the cell wait and continue.
MAX_FAILED_REQUESTS_PER_RUN = 5
REQUEST_CONNECT_TIMEOUT_SECONDS = 20
REQUEST_READ_TIMEOUT_SECONDS = 180
API_BASE = "https://www.renewables.ninja/api/data"

## Load and check query locations

Read the representative query locations, convert numeric fields and check for unique identifiers and valid coordinates. Create a lookup table for the acquisition and import steps.


In [4]:
query_locations = pd.read_csv(QUERY_LOCATIONS_PATH)
for column in ["representative_lat", "representative_lon", "cluster_capacity_mw", "cluster_it_load_mw", "campus_count"]:
    if column in query_locations.columns:
        query_locations[column] = pd.to_numeric(query_locations[column], errors="coerce")

required_query_columns = {"query_location_id", "representative_lat", "representative_lon"}
if not required_query_columns.issubset(query_locations.columns):
    raise ValueError(f"Query locations are missing {sorted(required_query_columns - set(query_locations.columns))}")
if query_locations.empty or query_locations["query_location_id"].isna().any() or query_locations["query_location_id"].duplicated().any():
    raise ValueError("Query locations must have present, unique identifiers")
if not query_locations["representative_lat"].between(-90, 90).all() or not query_locations["representative_lon"].between(-180, 180).all():
    raise ValueError("Query-location coordinates are missing or outside valid bounds")
query_lookup = query_locations.set_index("query_location_id", drop=False)

## Match requests to cached responses

Define annual request parameters with six-decimal coordinates and compare them with the metadata in cached response headers. Reuse matching responses and choose a request-specific filename when an existing file contains different settings, preserving the existing response.


In [5]:
# Make a query identifier suitable for a response filename.
def safe_id(value):
    return str(value).replace("/", "-").replace(" ", "_").replace("|", "-")


# Define the inclusive annual request dates.
def year_period(weather_year):
    return int(weather_year), f"{int(weather_year)}-01-01", f"{int(weather_year)}-12-31"


# Combine technology settings with six-decimal coordinates and request dates.
def build_request_params(location, technology, date_from, date_to):
    params = dict(PV_PARAMS if technology == "pv" else WIND_PARAMS)
    params.update(
        {
            "lat": round(float(location["representative_lat"]), 6),
            "lon": round(float(location["representative_lon"]), 6),
            "date_from": date_from,
            "date_to": date_to,
        }
    )
    return params


# Read and validate request parameters stored in the response header.
def read_response_metadata(path):
    """Read the request identity from a raw response header without loading its data."""
    metadata = None
    with Path(path).open() as handle:
        for line in handle:
            if not line.strip():
                continue
            if not line.startswith("#"):
                break
            header = line[1:].strip()
            if header.startswith("{"):
                try:
                    metadata = json.loads(header)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON response header in {path}") from exc
    if not isinstance(metadata, dict) or not isinstance(metadata.get("params"), dict):
        raise ValueError(f"Missing request parameters in response header: {path}")
    return metadata


# Identify echoed parameters that differ from the requested settings.
def request_parameter_mismatches(metadata, expected):
    """Compare echoed numeric strings and the builder's actual request settings."""
    observed = metadata["params"]
    mismatches = []
    for key, value in expected.items():
        if key == "format":
            continue  # The service does not echo format; the parser checks CSV columns.
        actual = observed.get(key)
        if isinstance(value, (int, float)):
            try:
                matches = float(actual) == float(value)
            except (TypeError, ValueError):
                matches = False
        else:
            matches = str(actual).strip() == str(value).strip()
        if not matches:
            mismatches.append(key)
    return mismatches


# Choose a matching supplied response or a distinct target for another request.
def build_cache_path(location, technology, weather_year):
    """Reuse a matching supplied response; keep other requests in distinct files."""
    year, date_from, date_to = year_period(weather_year)
    expected = build_request_params(location, technology, date_from, date_to)
    stem = f"{safe_id(location['query_location_id'])}_{technology}_{year}"
    supplied_path = RAW_CACHE_DIR / f"{stem}.csv"
    signature = sha256(json.dumps(expected, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    request_path = RAW_CACHE_DIR / f"{stem}_{signature}.csv"
    if supplied_path.exists():
        metadata = read_response_metadata(supplied_path)
        if not request_parameter_mismatches(metadata, expected):
            return supplied_path
    if request_path.exists():
        mismatches = request_parameter_mismatches(read_response_metadata(request_path), expected)
        if mismatches:
            raise ValueError(f"Request-specific cache has incompatible parameters {mismatches}: {request_path}")
        return request_path
    return request_path if supplied_path.exists() else supplied_path

## Define request-plan checks

Define checks for exactly one request per location, technology and weather year, with matching dates and a separate response path for each request. Standardize saved progress fields and clear previous attempt details for pending requests.


In [6]:
# Require one identified response per current location, technology and year.
def validate_request_plan(request_plan, query_locations):
    """Require one request per current location, technology and configured year."""
    key = ["query_location_id", "technology", "weather_year"]
    required = set(key + ["date_from", "date_to", "cache_file"])
    if not required.issubset(request_plan.columns):
        raise ValueError(f"Request plan is missing {sorted(required - set(request_plan.columns))}")
    expected = {
        (query_id, technology, year)
        for query_id in query_locations["query_location_id"]
        for technology in TECHNOLOGIES
        for year in WEATHER_YEARS
    }
    actual = set(request_plan[key].itertuples(index=False, name=None))
    if not expected or request_plan[key].isna().any().any() or request_plan.duplicated(key).any() or actual != expected:
        raise ValueError("Request plan must cover each current location/technology/year exactly once")
    if request_plan["cache_file"].isna().any() or request_plan["cache_file"].duplicated().any():
        raise ValueError("Every request needs its own raw response path")
    for request in request_plan.itertuples(index=False):
        _, date_from, date_to = year_period(request.weather_year)
        if request.date_from != date_from or request.date_to != date_to:
            raise ValueError(f"Request dates do not match configured year for {request.query_location_id}")

REQUEST_PLAN_STRING_COLUMNS = [
    "request_status",
    "request_mode",
    "request_started_at",
    "request_finished_at",
    "error_message",
    "notes",
]


# Timestamp request progress in UTC.
def utcnow_iso():
    return pd.Timestamp.now(tz="UTC").isoformat()


# Normalize progress fields and clear stale attempt metadata for pending requests.
def normalize_request_plan_types(request_plan):
    request_plan = request_plan.copy()
    for column in REQUEST_PLAN_STRING_COLUMNS:
        if column not in request_plan.columns:
            request_plan[column] = pd.NA
        request_plan[column] = request_plan[column].astype("object")
    if "http_status" not in request_plan.columns:
        request_plan["http_status"] = pd.NA
    request_plan["http_status"] = request_plan["http_status"].astype("object")

    pending = request_plan["request_status"].fillna("").eq("pending")
    request_plan.loc[pending, ["request_started_at", "request_finished_at", "error_message", "http_status"]] = pd.NA
    return request_plan

## Build the request plan

Create all location, technology and year combinations and match each request to its response path. Retain matching progress records, identify cached responses and save the request plan.


In [7]:
previous_plan = normalize_request_plan_types(pd.read_csv(REQUEST_PLAN_PATH)) if REQUEST_PLAN_PATH.exists() else pd.DataFrame()
previous_lookup = {}
if not previous_plan.empty:
    for _, row in previous_plan.iterrows():
        key = (
            row.get("query_location_id"),
            row.get("technology"),
            int(row.get("weather_year")) if pd.notna(row.get("weather_year")) else row.get("weather_year"),
            row.get("request_mode", "annual"),
        )
        previous_lookup[key] = row.to_dict()

request_rows = []
for _, location in query_locations.sort_values("query_location_id").iterrows():
    for weather_year in WEATHER_YEARS:
        _, date_from, date_to = year_period(weather_year)
        for technology in TECHNOLOGIES:
            cache_path = build_cache_path(location, technology, weather_year)
            key = (location["query_location_id"], technology, int(weather_year), "annual")
            previous = previous_lookup.get(key, {})
            if previous.get("cache_file") != str(cache_path.relative_to(PROJECT_ROOT)):
                previous = {}
            status = "pending"
            if cache_path.exists() and not FORCE_DOWNLOAD:
                status = previous.get("request_status") if previous.get("request_status") in {"downloaded", "cached"} else "cached"
            elif previous.get("request_status") == "rate_limited":
                status = "rate_limited"
            elif previous.get("request_status") == "failed" and not RETRY_FAILED:
                status = "failed"

            request_rows.append(
                {
                    "query_location_id": location["query_location_id"],
                    "country_timezone": location.get("country_timezone", ""),
                    "country_code": location.get("country_code", ""),
                    "country": location.get("country", ""),
                    "timezone": location.get("timezone", ""),
                    "renewable_query_region": location.get("renewable_query_region", ""),
                    "technology": technology,
                    "weather_year": int(weather_year),
                    "date_from": date_from,
                    "date_to": date_to,
                    "representative_lat": location["representative_lat"],
                    "representative_lon": location["representative_lon"],
                    "campus_count": location.get("campus_count", pd.NA),
                    "cluster_capacity_mw": location.get("cluster_capacity_mw", pd.NA),
                    "cluster_it_load_mw": location.get("cluster_it_load_mw", pd.NA),
                    "cache_file": str(cache_path.relative_to(PROJECT_ROOT)),
                    "request_mode": "annual",
                    "request_status": status,
                    "http_status": previous.get("http_status", pd.NA),
                    "request_started_at": previous.get("request_started_at", pd.NA),
                    "request_finished_at": previous.get("request_finished_at", pd.NA),
                    "error_message": previous.get("error_message", pd.NA),
                    "notes": "Annual request because Renewables.ninja enforces a maximum request date range of P1Y.",
                }
            )

request_plan = normalize_request_plan_types(pd.DataFrame(request_rows))
validate_request_plan(request_plan, query_locations)
request_plan.to_csv(REQUEST_PLAN_PATH, index=False)

print(f"Query locations: {len(query_locations):,}")
print(f"Annual Renewables.ninja requests: {len(request_plan):,}")
print(f"Minimum hourly batches at {HOURLY_REQUEST_LIMIT}/hour: {(len(request_plan) + HOURLY_REQUEST_LIMIT - 1) // HOURLY_REQUEST_LIMIT:,}")
request_plan["request_status"].value_counts(dropna=False)

Query locations: 87
Annual Renewables.ninja requests: 522
Minimum hourly batches at 50/hour: 11


request_status
cached    522
Name: count, dtype: int64

## Define rate-limit handling

Define helpers to count recent API attempts and calculate delays for the rolling hourly request limit. Use the service-reported retry delay when available and include a buffer before resuming requests.


In [8]:
# Count recent API attempts against the rolling hourly limit.
def recent_api_call_count(request_plan):
    if "request_finished_at" not in request_plan.columns:
        return 0
    finished = pd.to_datetime(request_plan["request_finished_at"], utc=True, errors="coerce")
    statuses = request_plan["request_status"].fillna("").isin(["downloaded", "failed", "rate_limited"])
    one_hour_ago = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=1)
    return int((statuses & (finished >= one_hour_ago)).sum())


# Determine whether the latest rate-limit response is still active.
def recent_rate_limit_hit(request_plan):
    if "request_finished_at" not in request_plan.columns or "request_status" not in request_plan.columns:
        return False, pd.NaT
    finished = pd.to_datetime(request_plan["request_finished_at"], utc=True, errors="coerce")
    rate_limited = request_plan["request_status"].fillna("").eq("rate_limited")
    if not rate_limited.any():
        return False, pd.NaT
    latest = finished[rate_limited].max()
    if pd.isna(latest):
        return False, pd.NaT
    return latest >= pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=1), latest


# Calculate the delay before another rolling-hour request slot is available.
def seconds_until_hourly_slot(request_plan):
    if "request_finished_at" not in request_plan.columns:
        return 0
    finished = pd.to_datetime(request_plan["request_finished_at"], utc=True, errors="coerce")
    counted = request_plan["request_status"].fillna("").isin(["downloaded", "failed", "rate_limited"])
    one_hour_ago = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=1)
    recent_finished = finished[counted & (finished >= one_hour_ago)]
    if len(recent_finished) < HOURLY_REQUEST_LIMIT:
        return 0
    oldest = recent_finished.min()
    wait_seconds = 3600 - (pd.Timestamp.now(tz="UTC") - oldest).total_seconds() + RATE_LIMIT_WAIT_BUFFER_SECONDS
    return max(0, int(wait_seconds) + 1)


# Extract a retry delay supplied in an API error message.
def parse_rate_limit_wait_seconds(error_message):
    match = re.search(r"Expected available in (\d+) seconds", str(error_message or ""))
    if match:
        return int(match.group(1)) + RATE_LIMIT_WAIT_BUFFER_SECONDS
    return None


# Calculate the remaining retry wait from the recorded response and delay.
def seconds_until_rate_limit_retry(request_plan, latest_rate_limit_at):
    if pd.isna(latest_rate_limit_at):
        return RATE_LIMIT_WAIT_BUFFER_SECONDS
    latest_rate_limit_at = pd.Timestamp(latest_rate_limit_at)
    if latest_rate_limit_at.tzinfo is None:
        latest_rate_limit_at = latest_rate_limit_at.tz_localize("UTC")
    elapsed = (pd.Timestamp.now(tz="UTC") - latest_rate_limit_at).total_seconds()

    rate_limited = request_plan["request_status"].fillna("").eq("rate_limited")
    finished = pd.to_datetime(request_plan["request_finished_at"], utc=True, errors="coerce")
    matching_rows = request_plan.loc[rate_limited & finished.eq(latest_rate_limit_at)]
    parsed_waits = [parse_rate_limit_wait_seconds(value) for value in matching_rows.get("error_message", pd.Series(dtype="object"))]
    parsed_waits = [value for value in parsed_waits if value is not None]
    if parsed_waits:
        return max(0, int(max(parsed_waits) - elapsed) + 1)

    wait_seconds = 3600 - elapsed + RATE_LIMIT_WAIT_BUFFER_SECONDS
    return max(0, int(wait_seconds) + 1)


# Explain and apply the required acquisition wait.
def wait_for_rate_limit_window(wait_seconds, reason):
    if wait_seconds <= 0:
        return
    print(f"Waiting {wait_seconds} seconds before continuing ({reason}).")
    time.sleep(wait_seconds)

## Define profile acquisition

Define helpers to load private credentials only when a network request is needed and to reuse cached responses unless forced downloads are enabled. Record whether each request succeeds, fails or reaches the service rate limit.


In [9]:
# Load optional token settings without replacing existing environment values.
def load_env_file(path):
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


# Reuse a cached response or fetch the requested profile with a private token.
def download_profile(location, technology, date_from, date_to, cache_path):
    if cache_path.exists() and not FORCE_DOWNLOAD:
        return "cached", None, None
    load_env_file(PROJECT_ROOT / ".env")
    api_token = os.getenv("RENEWABLES_NINJA_API_TOKEN") or os.getenv("RN_API_TOKEN")
    if not api_token:
        raise ValueError("Set RENEWABLES_NINJA_API_TOKEN or RN_API_TOKEN in .env or the shell environment before downloading.")

    headers = {"Authorization": f"Token {api_token}"}
    url = f"{API_BASE}/{technology}"
    params = build_request_params(location, technology, date_from, date_to)
    try:
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=(REQUEST_CONNECT_TIMEOUT_SECONDS, REQUEST_READ_TIMEOUT_SECONDS),
        )
    except requests.exceptions.RequestException as exc:
        return "failed", f"Request exception: {type(exc).__name__}: {exc}", None

    if response.status_code == 429:
        return "rate_limited", f"HTTP 429: {response.text[:500]}", response.status_code
    if response.status_code != 200:
        return "failed", f"HTTP {response.status_code}: {response.text[:500]}", response.status_code

    cache_path.write_text(response.text)
    return "downloaded", None, response.status_code

## Resume profile acquisition

Reload the saved request plan and process pending or rate-limited requests while respecting the configured waiting periods and request limits. Save progress after each attempt so acquisition can resume from the remaining requests.


In [10]:
request_plan = normalize_request_plan_types(pd.read_csv(REQUEST_PLAN_PATH))
validate_request_plan(request_plan, query_locations)

active_rate_limit, latest_rate_limit_at = recent_rate_limit_hit(request_plan)
if active_rate_limit:
    wait_seconds = seconds_until_rate_limit_retry(request_plan, latest_rate_limit_at)
    if AUTO_WAIT_ON_RATE_LIMIT:
        wait_for_rate_limit_window(wait_seconds, f"previous HTTP 429 at {latest_rate_limit_at}")
    else:
        raise RuntimeError(
            f"Previous HTTP 429 was recorded at {latest_rate_limit_at}. Wait until the one-hour window has passed, then rerun this cell."
        )

last_request_monotonic = None
new_api_calls = 0
new_failures = 0

for index, request in request_plan.iterrows():
    request_status = str(request.get("request_status", ""))
    if request_status not in {"pending", "rate_limited"}:
        continue

    cache_path = PROJECT_ROOT / str(request["cache_file"])
    if cache_path.exists() and not FORCE_DOWNLOAD:
        request_plan.loc[index, "request_status"] = "cached"
        request_plan.loc[index, "error_message"] = pd.NA
        request_plan.to_csv(REQUEST_PLAN_PATH, index=False)
        print(f"{request['query_location_id']} {request['technology']} {request['date_from']} to {request['date_to']}: cached")
        continue

    while True:
        if MAX_NEW_REQUESTS_PER_RUN is not None and new_api_calls >= MAX_NEW_REQUESTS_PER_RUN:
            print(f"Stopping: reached MAX_NEW_REQUESTS_PER_RUN={MAX_NEW_REQUESTS_PER_RUN}.")
            break

        if new_failures >= MAX_FAILED_REQUESTS_PER_RUN:
            print(f"Stopping: reached MAX_FAILED_REQUESTS_PER_RUN={MAX_FAILED_REQUESTS_PER_RUN}. Check error messages before rerunning.")
            break

        if recent_api_call_count(request_plan) >= HOURLY_REQUEST_LIMIT:
            wait_seconds = seconds_until_hourly_slot(request_plan)
            if AUTO_WAIT_ON_RATE_LIMIT:
                wait_for_rate_limit_window(wait_seconds, f"rolling {HOURLY_REQUEST_LIMIT}/hour request limit")
            else:
                print(f"Stopping: reached rolling hourly limit of {HOURLY_REQUEST_LIMIT} API calls.")
                break

        if last_request_monotonic is not None:
            elapsed = time.monotonic() - last_request_monotonic
            sleep_seconds = max(0.0, MIN_SECONDS_BETWEEN_REQUESTS - elapsed)
            if sleep_seconds > 0:
                time.sleep(sleep_seconds)

        location = query_lookup.loc[request["query_location_id"]]
        request_plan.loc[index, "request_status"] = "pending"
        request_plan.loc[index, "request_started_at"] = utcnow_iso()
        request_plan.to_csv(REQUEST_PLAN_PATH, index=False)

        status, error, http_status = download_profile(
            location,
            request["technology"],
            request["date_from"],
            request["date_to"],
            cache_path,
        )
        last_request_monotonic = time.monotonic()
        new_api_calls += 1

        request_plan.loc[index, "request_status"] = status
        request_plan.loc[index, "error_message"] = error
        request_plan.loc[index, "http_status"] = http_status
        request_plan.loc[index, "request_finished_at"] = utcnow_iso()
        request_plan.to_csv(REQUEST_PLAN_PATH, index=False)

        if status == "failed":
            new_failures += 1
        print(f"{request['query_location_id']} {request['technology']} {request['date_from']} to {request['date_to']}: {status}")

        if status == "rate_limited" or http_status == 429:
            if AUTO_WAIT_ON_RATE_LIMIT:
                latest_finished = pd.to_datetime(request_plan.loc[index, "request_finished_at"], utc=True, errors="coerce")
                wait_seconds = seconds_until_rate_limit_retry(request_plan, latest_finished)
                wait_for_rate_limit_window(wait_seconds, "HTTP 429 rate-limit response")
                continue
            print("Stopping: API returned HTTP 429 rate-limit response. Rerun this cell after the one-hour window has passed.")
            break

        break

    if MAX_NEW_REQUESTS_PER_RUN is not None and new_api_calls >= MAX_NEW_REQUESTS_PER_RUN:
        break
    if new_failures >= MAX_FAILED_REQUESTS_PER_RUN:
        break

print(f"New API calls in this run: {new_api_calls}")
print(f"New failures in this run: {new_failures}")
request_plan["request_status"].value_counts(dropna=False)

New API calls in this run: 0
New failures in this run: 0


request_status
cached    522
Name: count, dtype: int64

## Define response validation

Define a parser that checks request parameters, unique UTC timestamps, complete hourly coverage and finite technology values. Retain the validated UTC and local timestamps alongside each technology's capacity factor.


In [11]:
# Validate raw hourly observations before selecting the technology values.
def parse_ninja_csv(path, technology, expected_params):
    """Validate the raw response before selecting its canonical hourly values."""
    metadata = read_response_metadata(path)
    mismatches = request_parameter_mismatches(metadata, expected_params)
    if mismatches:
        raise ValueError(f"Cached response does not match requested parameters {mismatches}: {path}")
    data_lines = [line for line in Path(path).read_text().splitlines() if line.strip() and not line.startswith("#")]
    if not data_lines:
        raise ValueError(f"No CSV data lines found in {path}")
    frame = pd.read_csv(StringIO(chr(10).join(data_lines)))
    if "time" not in frame.columns or "electricity" not in frame.columns:
        raise ValueError(f"Unexpected Renewables.ninja columns in {path}: {list(frame.columns)}")

    frame["time_utc"] = pd.to_datetime(frame["time"], utc=True, errors="coerce")
    if frame["time_utc"].isna().any():
        raise ValueError(f"Invalid raw UTC timestamps in {path}")
    if frame["time_utc"].duplicated().any():
        raise ValueError(f"Duplicate raw UTC timestamps in {path}")
    output_column = "solar_capacity_factor" if technology == "pv" else "wind_capacity_factor"
    frame[output_column] = pd.to_numeric(frame["electricity"], errors="coerce")
    if not np.isfinite(frame[output_column].to_numpy(dtype=float)).all():
        raise ValueError(f"Missing or nonfinite {technology} values in {path}")

    expected_hours = pd.date_range(
        pd.Timestamp(expected_params["date_from"], tz="UTC"),
        pd.Timestamp(expected_params["date_to"], tz="UTC") + pd.Timedelta(days=1),
        freq="h", inclusive="left",
    )
    if not pd.DatetimeIndex(frame["time_utc"]).sort_values().equals(expected_hours):
        raise ValueError(f"Response does not contain the complete requested UTC hourly calendar: {path}")
    frame["time_local"] = pd.to_datetime(frame["local_time"], errors="coerce") if "local_time" in frame.columns else pd.NaT
    return frame[["time_utc", "time_local", output_column]]

## Combine solar and wind profiles

Import the validated annual responses for each location and align solar and wind values on the same UTC calendar. Calculate their 50/50 mean as the renewable-availability proxy and retain the contributing source filenames.


In [12]:
request_plan = normalize_request_plan_types(pd.read_csv(REQUEST_PLAN_PATH))
validate_request_plan(request_plan, query_locations)
missing_files = [str(path) for path in (PROJECT_ROOT / str(value) for value in request_plan["cache_file"]) if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} requested renewable responses are missing. Resume acquisition before import. Examples: {missing_files[:3]}"
    )

hourly_frames = []
for query_location_id in query_locations["query_location_id"].sort_values():
    location = query_lookup.loc[query_location_id]
    parsed_by_technology = {}
    source_files = {}
    for technology in TECHNOLOGIES:
        planned = request_plan.loc[
            request_plan["query_location_id"].eq(query_location_id) & request_plan["technology"].eq(technology)
        ].sort_values("weather_year")
        frames = []
        paths = []
        for request in planned.itertuples(index=False):
            path = PROJECT_ROOT / str(request.cache_file)
            params = build_request_params(location, technology, request.date_from, request.date_to)
            frames.append(parse_ninja_csv(path, technology, params))
            paths.append(path)
        parsed_by_technology[technology] = pd.concat(frames, ignore_index=True).sort_values("time_utc")
        source_files[technology] = "|".join(str(path.relative_to(PROJECT_ROOT)) for path in paths)

    merged = parsed_by_technology["pv"].merge(
        parsed_by_technology["wind"], on="time_utc", how="outer", suffixes=("_pv", "_wind"), validate="one_to_one",
    )
    if merged[["solar_capacity_factor", "wind_capacity_factor"]].isna().any().any():
        raise ValueError(f"PV and wind calendars do not match for {query_location_id}")
    merged["time_local"] = merged.get("time_local_pv", pd.Series(pd.NaT, index=merged.index)).combine_first(
        merged.get("time_local_wind", pd.Series(pd.NaT, index=merged.index))
    )
    merged["weather_year"] = merged["time_utc"].dt.year
    merged = merged[merged["weather_year"].isin(WEATHER_YEARS)].copy()
    merged["query_location_id"] = query_location_id
    merged["country_code"] = location.get("country_code", "")
    merged["country"] = location.get("country", "")
    merged["timezone"] = location.get("timezone", "")
    merged["renewable_query_region"] = location.get("renewable_query_region", "")
    merged["representative_lat"] = location.get("representative_lat", pd.NA)
    merged["representative_lon"] = location.get("representative_lon", pd.NA)
    merged["campus_count"] = location.get("campus_count", pd.NA)
    merged["cluster_capacity_mw"] = location.get("cluster_capacity_mw", pd.NA)
    merged["cluster_it_load_mw"] = location.get("cluster_it_load_mw", pd.NA)
    merged["renewable_availability_index"] = merged[["solar_capacity_factor", "wind_capacity_factor"]].mean(axis=1, skipna=False)
    merged["pv_source_file"] = source_files.get("pv", "")
    merged["wind_source_file"] = source_files.get("wind", "")
    hourly_frames.append(merged)

## Validate and save hourly profiles

Check that the combined table covers every requested location and weather year without duplicate UTC keys or nonfinite values. Save the validated hourly profiles as Parquet and display their coverage.


In [13]:
output_columns = [
    "query_location_id",
    "country_code",
    "country",
    "timezone",
    "renewable_query_region",
    "representative_lat",
    "representative_lon",
    "campus_count",
    "cluster_capacity_mw",
    "cluster_it_load_mw",
    "weather_year",
    "time_utc",
    "time_local",
    "solar_capacity_factor",
    "wind_capacity_factor",
    "renewable_availability_index",
    "pv_source_file",
    "wind_source_file",
]

hourly = pd.concat(hourly_frames, ignore_index=True)
hourly = hourly[output_columns].sort_values(["query_location_id", "time_utc"]).reset_index(drop=True)
hours_per_location = sum(
    len(pd.date_range(f"{year}-01-01", f"{year + 1}-01-01", freq="h", inclusive="left", tz="UTC"))
    for year in WEATHER_YEARS
)
if len(hourly) != len(query_locations) * hours_per_location:
    raise ValueError("Imported renewable hours do not cover the current query/year calendar")
if hourly.duplicated(["query_location_id", "time_utc"]).any():
    raise ValueError("Imported renewable data contain duplicate query-location UTC keys")
if not np.isfinite(hourly[["solar_capacity_factor", "wind_capacity_factor", "renewable_availability_index"]].to_numpy(dtype=float)).all():
    raise ValueError("Hourly PV, wind and renewable-index values must be finite")

# Normalize numeric representation before storage for deterministic downstream use.
for column in ["solar_capacity_factor", "wind_capacity_factor", "renewable_availability_index"]:
    hourly[column] = pd.to_numeric(hourly[column].astype(str), errors="raise")
hourly.to_parquet(HOURLY_OUTPUT_PATH, index=False, compression="zstd")

print(f"Complete query locations imported: {hourly['query_location_id'].nunique():,} / {len(query_locations):,}")
print(f"Hourly rows: {len(hourly):,}")
print(hourly.groupby("weather_year")["time_utc"].nunique())
hourly.head()

Complete query locations imported: 87 / 87
Hourly rows: 2,288,448
weather_year
2023    8760
2024    8784
2025    8760
Name: time_utc, dtype: int64


,query_location_id,country_code,country,timezone,renewable_query_region,representative_lat,representative_lon,campus_count,cluster_capacity_mw,cluster_it_load_mw,weather_year,time_utc,time_local,solar_capacity_factor,wind_capacity_factor,renewable_availability_index,pv_source_file,wind_source_file
0,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,2023,2023-01-01 00:00:00+00:00,2023-01-01 04:00:00,0.000,0.092,0.0460,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
1,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,2023,2023-01-01 01:00:00+00:00,2023-01-01 05:00:00,0.000,0.132,0.0660,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
2,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,2023,2023-01-01 02:00:00+00:00,2023-01-01 06:00:00,0.000,0.167,0.0835,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
3,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,2023,2023-01-01 03:00:00+00:00,2023-01-01 07:00:00,0.071,0.169,0.1200,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....
4,RN-ARE-DUBAI-01,ARE,United Arab Emirates,Asia/Dubai,ARE Asia/Dubai cluster 01,24.631681,54.776669,3,60.685811,52.77027,2023,2023-01-01 04:00:00+00:00,2023-01-01 08:00:00,0.292,0.128,0.2100,data/raw/renewables/RN-ARE-DUBAI-01_pv_2023.cs...,data/raw/renewables/RN-ARE-DUBAI-01_wind_2023....


## Inspect capacity-factor distributions

Summarize the solar, wind and combined-index values to inspect their ranges and distributions across all imported hours.


In [14]:
display(hourly[["solar_capacity_factor", "wind_capacity_factor", "renewable_availability_index"]].describe())

,solar_capacity_factor,wind_capacity_factor,renewable_availability_index
count,2.288448e+06,2.288448e+06,2.288448e+06
mean,1.682510e-01,3.088405e-01,2.385457e-01
std,2.427192e-01,2.735133e-01,1.720555e-01
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,7.300000e-02,9.800000e-02
50%,0.000000e+00,2.340000e-01,2.125000e-01
75%,3.120000e-01,4.900000e-01,3.530000e-01
max,9.000000e-01,9.910000e-01,9.405000e-01
